In [ ]:
import numpy as np
from dataclasses import dataclass

In [ ]:
@dataclass
class Node:
    coords: tuple[float, float]
    value: float

num_nodes = 200
x_lim = (0.0, 0.5)
y_lim = (0.0, 0.5)
nodes = []
for _ in range(num_nodes):
    rand_x = np.random.uniform(x_lim[0], x_lim[1])
    rand_y = np.random.uniform(y_lim[0], y_lim[1])
    value = 1.0 - (rand_x - rand_y)
    nodes.append(Node(coords=(rand_x, rand_y), value=value))

adj_matrix = np.zeros((num_nodes, num_nodes))
for i in range(num_nodes):
    for j in range(num_nodes):
        if i != j:
            dist = np.linalg.norm(np.array(nodes[i].coords) - np.array(nodes[j].coords))
            if dist < 0.1:
                adj_matrix[i, j] = 1.0 / (1 + dist)


In [ ]:
import matplotlib.pyplot as plt

for i in range(num_nodes):
    for j in range(i + 1, num_nodes):
        if adj_matrix[i, j] > 0:
            x_coords = [nodes[i].coords[0], nodes[j].coords[0]]
            y_coords = [nodes[i].coords[1], nodes[j].coords[1]]
            plt.plot(x_coords, y_coords, c='gray', alpha=0.5)
x_coords = [node.coords[0] for node in nodes]
y_coords = [node.coords[1] for node in nodes]
values = [node.value for node in nodes]
plt.scatter(x_coords, y_coords, c=values, cmap='viridis')
plt.colorbar()
plt.show()


In [ ]:
K = 0.15
d = 0.35
h = 0.1
num_neighbors = np.sum(adj_matrix, axis=1)
omega = np.random.normal(size=num_nodes)
def dudt(states, adj_matrix):
    state_diff = states[None, :] - states[:, None]
    laplacian = d * np.sum(state_diff * adj_matrix, axis=1)

    sin_diff = np.sin(state_diff)
    coupling = K / num_neighbors * np.sum(adj_matrix * sin_diff, axis=1)

    return omega + h * np.sin(states) + coupling + np.random.normal(size=states.shape[0]) 

timesteps = 1000
dt = 0.01
states = np.array([node.value for node in nodes])

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 50.0

fig, ax = plt.subplots()
x_coords = [node.coords[0] for node in nodes]
y_coords = [node.coords[1] for node in nodes]
scat = ax.scatter(x_coords, y_coords, c=[node.value for node in nodes], cmap='viridis')
def update(frame):
    global states
    states = states + dt * dudt(states, adj_matrix)
    scat.set_array(states)
    return scat,
ani = animation.FuncAnimation(fig, func=update, frames=timesteps, blit=True)
HTML(ani.to_jshtml())